# The Gibbs Sampler

Discrete sampling and its variants all generate samples independently. A new sample in the trace has no knowledge of any previous samples that have been generated. A very common alternative treats the sampling as a *Markov Chain* in which the generation of the next sample depends on the current sample, but not on events prior to the current sample. Markov Chain Monte Carlo (MCMC) sampling methods generate each successive sample by randomly modifying the current sample. There are many different MCMC methods but one that is well-suited for Bayesian networks is the **Gibbs Sampler** named after the Statistical Physicist Josiah Willard Gibbs, whose work in statistical thermodynamics has some close parallels with (and significantly predates) this approach to random sampling.

Gibbs sampling is conceptually very simple.

* Begin by fixing all *evidence variables* to their observed values. These will not be changed through the sampling process.
* Initialise all of the non-evidence variables to random values. These initial values must be supported by the distribution attached to the variable but there are no other requirement of this initialisation.
* A new state is created by sampling *one* of the non-evidence variables, chosen at random. The sampling of the chosen variable will be from its conditional probability distribution which is implicitly or explicitly conditioned on its *Markov Blanket* which is the set of variables that are either parents or children of the chosen variable, or are co-parents of the variable's children. It can be proven that any variable is conditionally independent of all variables outside of its Markov blanket and so we only need to know the local neighbourhood. For small systems, this concept seems rather unecessary, but for Bayesian networks with large number of variables this notion helps to improve the efficiency of the computation.
* The Gibbs sampler determines its outcome by counting how many times it visits each state and normalising. If it visits a state where $M$ is True forty-five times, and $M$ is False fifteen times, the resulting distribution is $P(M\vert F,P)=(0.25,0.75)$. 

The Gibbs sampler is a classical Markov Chain and can be analysed in terms of dynamical systems theory. We will not go in to this in detail here, but we will state the main finding of the analysis: the sampling process settles into an equilibrium in which the the long-run fraction of time that the sampler spends in each state is exactly proportional to that state's posterior probability. This results from the transition probabilities between states being defined by the Markov blankets of the non-evidence variables.

An important aspect of working with the Gibbs samples is that it needs to settle down to that equilibrium state to yield accurate results. In the early stages of the sampling, the samples are not necessarily drawn in proportion to their posterior probability because the initially random state is not yet fully consistent with the sampling distribution. It is therefore very common for Gibbs samplers to require a *burn-in* period to ensure they have reached equilibrium before we start counting states.

We do not typically implement Gibbs samplers from first principles as they are quite tricky to implement and can be sensitive to numerical precision issues. Fortunately the library `pymc` provides us with a tried-and-trusted implementation that we can just use. Let's build a network in PYMC3.

New env
Install Jupyter and pymc

In [6]:
import numpy as np
import pymc as mc


PG = np.array([0.6,0.4])
PM = np.array([0.75,0.25])
PF_M = np.array([[0.8,0.4],[0.2,0.6]])
PP_M = np.array([[0.65,0.45],[0.35,0.55]]) 

with mc.Model() as model:
    G = mc.Bernoulli('G',PG[1])
    M = mc.Bernoulli('M',PM[1])
    P_F = mc.Deterministic('P_F', mc.math.switch(M,PF_M[1,1],PF_M[1,0]))
    F = mc.Bernoulli('F', P_F)
    P_P = mc.Deterministic('P_P', mc.math.switch(M,PP_M[1,1],PP_M[1,0]))
    P = mc.Bernoulli('P',P_P)
    step = mc.Metropolis()
    trace = mc.sample(10000, step=step, tune=5000, random_seed=123, progressbar=True)

AttributeError: module 'numpy' has no attribute '_core'

In [ ]:
mc.plot_trace(trace)

In [ ]:
for t in trace.posterior:
  print(t)

In [ ]:
df=trace.to_dataframe(groups='posterior')
print(df)

In [ ]:
P_Math_Finance = float(df[(df['Finance'] == 1) & (df['Math'] ==1)].shape[0]) / df[df['Finance'] == 1].shape[0]
print(P_Math_Finance)